<a href="https://colab.research.google.com/github/AngelaOrtiz25/tesis_angela/blob/gh-pages/datos_codigos/Dijkstra.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
!pip install pandas networkx
!pip install pandas

Una vez que se obtuvo la lista de los primeros quince mucnicipios conectados mendiante el pagerank se realiza el **Dijkstra**

In [11]:
import pandas as pd
import networkx as nx
from google.colab import drive

# Montar Google Drive
drive.mount('/content/drive')

# 1. CARGAR TUS DATOS DE RUTAS
df_rutas = pd.read_csv('/content/drive/MyDrive/tesis/distancias_chiapas_carreteras_3d_completolimpio.csv')

# 2. CARGAR TOP 10 DE PAGE RANK
df_top10 = pd.read_csv('/content/drive/MyDrive/tesis/top10_pagerank.csv')
destinos_prioritarios = df_top10['Municipio'].tolist()

print(f"Destinos prioritarios desde PageRank: {destinos_prioritarios}")

# 3. CONSTRUIR EL GRAFO
G = nx.DiGraph()

for _, row in df_rutas.iterrows():
    G.add_edge(
        row['Origen'],
        row['Destino'],
        weight=row['Distancia 3D (km)'],
        dist_2d=row['Distancia carretera (km)'],
        tiempo=row['Tiempo estimado (min)']
    )

# 4. ORIGEN FIJO
origen = "Tuxtla Gutierrez"

# 5. CALCULAR RUTAS CON DIJKSTRA
resultados = []

for destino in destinos_prioritarios:
    try:
        ruta_nodos = nx.shortest_path(G, source=origen, target=destino, weight='weight')

        dist_3d_total = 0
        dist_2d_total = 0
        tiempo_total = 0

        for i in range(len(ruta_nodos) - 1):
            u = ruta_nodos[i]
            v = ruta_nodos[i+1]
            datos = G[u][v]
            dist_3d_total += datos['weight']
            dist_2d_total += datos['dist_2d']
            tiempo_total += datos['tiempo']

        incremento_pct = ((dist_3d_total - dist_2d_total) / dist_2d_total) * 100

        resultados.append({
            'Municipio Destino': destino,
            'Ruta Óptima': ' → '.join(ruta_nodos),
            'Distancia 2D (km)': round(dist_2d_total, 1),
            'Distancia 3D (km)': round(dist_3d_total, 1),
            'Incremento Topográfico (%)': round(incremento_pct, 1),
            'Tiempo Est. (min)': round(tiempo_total, 0)
        })

    except nx.NetworkXNoPath:
        print(f"No hay ruta de {origen} a {destino}")

# 6. CREAR Y GUARDAR TABLA FINAL EN TU DRIVE
df_resultados = pd.DataFrame(resultados)

# Ruta completa en tu Google Drive
ruta_salida = '/content/drive/MyDrive/tesis/rutas_optimas_top15distraj.csv'
df_resultados.to_csv(ruta_salida, index=False, encoding='utf-8')

print("\n Resultados de Dijkstra:")
print(df_resultados.to_markdown(index=False))
print(f"\n✓ Resultados guardados de forma segura en: {ruta_salida}")

Mounted at /content/drive
Destinos prioritarios desde PageRank: ['San Cristobal de las Casas', 'Rayon', 'Chamula', 'Chiapilla', 'Pueblo Nuevo Solistahuacan', 'Aldama', 'Rincon Chamula San Pedro', 'Pantepec', 'Santiago el Pinar', 'Larrainzar', 'Teopisca', 'Totolapa', 'Tapilula', 'Acala', 'Mitontic']

 Resultados de Dijkstra:
| Municipio Destino          | Ruta Óptima                                   |   Distancia 2D (km) |   Distancia 3D (km) |   Incremento Topográfico (%) |   Tiempo Est. (min) |
|:---------------------------|:----------------------------------------------|--------------------:|--------------------:|-----------------------------:|--------------------:|
| San Cristobal de las Casas | Tuxtla Gutierrez → San Cristobal de las Casas |                61.4 |                61.9 |                          0.8 |                  56 |
| Rayon                      | Tuxtla Gutierrez → Copainala → Rayon          |               126.5 |               128.3 |                        